In [4]:
import zstandard as zstd
import json
from datetime import datetime

COMMENTS_PATH = r"D:\wallstreetbets\reddit\subreddits\wallstreetbets_comments.zst"

dctx = zstd.ZstdDecompressor(max_window_size=2**31)
with open(COMMENTS_PATH, 'rb') as f:
    with dctx.stream_reader(f) as reader:
        buffer = b''
        chunk_n = 0
        while True:
            chunk = reader.read(2**22)
            if not chunk:
                break
            chunk_n += 1
            buffer += chunk
            lines = buffer.split(b'\n')
            buffer = lines[-1]
            # 每100个chunk打印一次时间戳
            if chunk_n % 100 == 0:
                for line in reversed(lines[:-1]):
                    if line.strip():
                        try:
                            obj = json.loads(line)
                            ts = float(obj.get('created_utc', 0))
                            dt = datetime.utcfromtimestamp(ts)
                            print(f"Chunk {chunk_n}: {dt}")
                            break
                        except:
                            continue
            # 到2021年就停
            if chunk_n > 8000:
                break

Chunk 100: 2017-02-23 16:38:29
Chunk 200: 2018-02-10 21:39:04
Chunk 300: 2018-07-05 12:31:37
Chunk 400: 2018-09-13 18:40:52
Chunk 500: 2018-10-30 23:57:48
Chunk 600: 2018-12-24 15:46:47
Chunk 700: 2019-02-14 14:24:34
Chunk 800: 2019-04-02 08:21:59
Chunk 900: 2019-05-12 02:46:47
Chunk 1000: 2019-06-25 19:47:45
Chunk 1100: 2019-08-05 12:45:01
Chunk 1200: 2019-08-30 19:18:26
Chunk 1300: 2019-10-03 14:02:11
Chunk 1400: 2019-11-01 20:51:47
Chunk 1500: 2019-12-03 22:04:18
Chunk 1600: 2020-01-09 21:10:17
Chunk 1700: 2020-01-30 03:58:25
Chunk 1800: 2020-02-10 20:38:15
Chunk 1900: 2020-02-21 15:06:39
Chunk 2000: 2020-02-28 18:23:08
Chunk 2100: 2020-03-05 18:10:46
Chunk 2200: 2020-03-11 01:42:39
Chunk 2300: 2020-03-14 00:23:48
Chunk 2400: 2020-03-17 15:13:49
Chunk 2500: 2020-03-20 04:44:05
Chunk 2600: 2020-03-24 03:35:52
Chunk 2700: 2020-03-27 04:04:33
Chunk 2800: 2020-04-01 16:18:30
Chunk 2900: 2020-04-07 16:16:02
Chunk 3000: 2020-04-14 04:32:11
Chunk 3100: 2020-04-18 22:14:17
Chunk 3200: 2020-

In [7]:
"""
WSB GME Peak Period Word Cloud Generator
生成两张词云：submissions 和 comments 各一张
Jan 20 – Feb 10 2021，只保留 GME 相关内容

用法: python wsb_gme_wordcloud.py
"""

import zstandard as zstd
import json, re
from datetime import datetime, timezone
from pathlib import Path
from wordcloud import WordCloud, STOPWORDS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─── 配置 ───────────────────────────────────────────────────────────────
SUBMISSIONS_PATH = r"D:\wallstreetbets\reddit\subreddits\wallstreetbets_submissions.zst"
COMMENTS_PATH    = r"D:\wallstreetbets\reddit\subreddits\wallstreetbets_comments.zst"
OUTPUT_DIR       = r"D:\phd\during the GameStop\1. data"

START_TS = datetime(2021, 1, 20, tzinfo=timezone.utc).timestamp()
END_TS   = datetime(2021, 2, 25, tzinfo=timezone.utc).timestamp()

# 跳过早期 chunks 定位到 2021（submissions 和 comments 大小不同，分别配置）
SKIP_SUBMISSIONS = 325
SKIP_COMMENTS    = 7100

# GME 关键词过滤
GME_KEYWORDS = {
    'gme', 'gamestop', 'game stop', 'gamestock', '$gme',
    'melvin', 'short squeeze', 'short seller', 'citadel',
    'robinhood', 'moon', 'diamond hands', 'hold the line',
    'to the moon', 'apes', 'tendies',
}

# 停用词
EXTRA_STOPWORDS = {
    'http', 'https', 'www', 'com', 'reddit', 'subreddit', 'deleted', 'removed',
    'edit', 'amp', 'gt', 'lt', 'just', 'like', 'get', 'one', 'think', 'know',
    'go', 'going', 'got', 'also', 'would', 'could', 'will', 'use', 'used',
    'using', 'make', 'made', 'really', 'even', 'still', 'back', 'way', 'much',
    'many', 've', 're', 'll', 'don', 'doesn', 'didn', 'isn', 'aren', 'can',
    't', 's', 'im', 'ive', 'x200b', 'nan', 'none', 'post', 'comment', 'people',
    'time', 'day', 'right', 'need', 'see', 'lot', 'good', 'new', 'want', 'said',
    'say', 'thing', 'guys', 'well', 'things', 'actually', 'probably',
    'already', 'never', 'always', 'every', 'doing', 'done', 'they', 'their',
    'them', 'this', 'that', 'with', 'have', 'from', 'been', 'were',
    'gamestop', 'game', 'stop', 'gme', 'stock', 'stocks', 'share', 'shares',
    # 保留 WSB meme 词: yolo, hodl, apes, tendies, squeeze, moon,
    # diamond hands, calls, puts, short, hedge, retard, autist, dd 等不在此列
}
SW = STOPWORDS.union(EXTRA_STOPWORDS)


# ─── 读取 zst ───────────────────────────────────────────────────────────
def read_zst(filepath, text_fields, skip_chunks=0):
    if not Path(filepath).exists():
        print(f"  [跳过] 找不到文件: {filepath}")
        return []

    texts = []
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)
    with open(filepath, 'rb') as f:
        with dctx.stream_reader(f) as reader:
            buffer = b''
            chunk_n = 0
            done = False
            while not done:
                chunk = reader.read(2**22)
                if not chunk:
                    break
                chunk_n += 1
                if chunk_n < skip_chunks:
                    continue
                buffer += chunk
                lines = buffer.split(b'\n')
                buffer = lines[-1]
                for line in lines[:-1]:
                    if not line.strip():
                        continue
                    try:
                        obj = json.loads(line)
                        ts = float(obj.get('created_utc', 0))
                        if ts > END_TS:
                            done = True
                            break
                        if ts < START_TS:
                            continue
                        parts = [obj.get(f, '') or '' for f in text_fields]
                        full = ' '.join(parts).lower()
                        if not any(kw in full for kw in GME_KEYWORDS):
                            continue
                        texts.append(full)
                    except (json.JSONDecodeError, ValueError):
                        continue

    print(f"  ✓ {Path(filepath).name}: {len(texts):,} GME-related records")
    return texts


# ─── 清洗文本 ────────────────────────────────────────────────────────────
def clean(texts):
    combined = ' '.join(texts)
    combined = re.sub(r'http\S+', ' ', combined)
    # 只删非字母数字字符，保留字母（包括全大写meme词）
    combined = re.sub(r'[^a-zA-Z\s]', ' ', combined)
    # 只删单字符词，保留 dd, yolo 等2字符以上的词
    combined = re.sub(r'\b\w{1}\b', ' ', combined)
    combined = re.sub(r'\s+', ' ', combined).strip()
    return combined


# ─── 生成并保存词云 ──────────────────────────────────────────────────────
def make_wordcloud(texts, title, subtitle, output_path):
    if not texts:
        print(f"  [跳过] 没有文本可生成: {title}")
        return

    combined = clean(texts)

    wc = WordCloud(
        width=1600,
        height=900,
        background_color='white',
        colormap='Blues',
        stopwords=SW,
        max_words=150,
        min_font_size=12,
        max_font_size=220,
        prefer_horizontal=0.85,
        collocations=False,
        relative_scaling=0.6,
    ).generate(combined)

    fig, ax = plt.subplots(figsize=(16, 9), facecolor='white')
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, color='#1a1a2e', fontsize=18, fontweight='bold', pad=16)
    fig.text(
        0.5, 0.01, subtitle,
        ha='center', color='gray', fontsize=9,
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  ✓ 已保存: {output_path}")


# ─── 主程序 ─────────────────────────────────────────────────────────────
def main():
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print("WSB GME Word Cloud  |  Jan 20 – Feb 25, 2021")
    print("=" * 60)

    # ── Submissions ──────────────────────────────────────────────
    print("\n[1/2] Reading submissions (title + selftext)...")
    sub_texts = read_zst(
        SUBMISSIONS_PATH,
        text_fields=['title', 'selftext'],
        skip_chunks=SKIP_SUBMISSIONS,
    )
    make_wordcloud(
        sub_texts,
        title='r/WallStreetBets — GME Submissions  (Jan 20 – Feb 25, 2021)',
        subtitle='Source: Academic Torrents  |  Filtered: GME/GameStop mentions only  |  title + selftext',
        output_path=str(output_dir / 'wsb_gme_wordcloud_submissions.png'),
    )

    # ── Comments ─────────────────────────────────────────────────
    print("\n[2/2] Reading comments (body)...")
    com_texts = read_zst(
        COMMENTS_PATH,
        text_fields=['body'],
        skip_chunks=SKIP_COMMENTS,
    )
    make_wordcloud(
        com_texts,
        title='r/WallStreetBets — GME Comments  (Jan 20 – Feb 25, 2021)',
        subtitle='Source: Academic Torrents  |  Filtered: GME/GameStop mentions only  |  comment body',
        output_path=str(output_dir / 'wsb_gme_wordcloud_comments.png'),
    )

    print("\n✓ 完成！两张词云已保存到:")
    print(f"  {output_dir}")


if __name__ == '__main__':
    main()

WSB GME Word Cloud  |  Jan 20 – Feb 25, 2021

[1/2] Reading submissions (title + selftext)...
  ✓ wallstreetbets_submissions.zst: 266,058 GME-related records
  ✓ 已保存: D:\phd\during the GameStop\1. data\wsb_gme_wordcloud_submissions.png

[2/2] Reading comments (body)...
  ✓ wallstreetbets_comments.zst: 1,262,129 GME-related records
  ✓ 已保存: D:\phd\during the GameStop\1. data\wsb_gme_wordcloud_comments.png

✓ 完成！两张词云已保存到:
  D:\phd\during the GameStop\1. data
